# 03 — Run Evaluation
Runs RAGAS across every (chunking × retrieval) combination and saves a comparison table.
Make sure 02_build_index.ipynb has been run for each chunking strategy first.

In [ ]:
# ── Environment & path setup ─────────────────────────────────────────────────
import os, sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent  # notebooks/ -> project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
QDRANT_URL     = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

print("OPENAI_API_KEY set:", bool(OPENAI_API_KEY))
print("QDRANT_URL       :", QDRANT_URL)

In [ ]:
# ── Clients ──────────────────────────────────────────────────────────────────
import torch
from openai import OpenAI
from qdrant_client import QdrantClient
from fastembed import SparseTextEmbedding
from sentence_transformers import CrossEncoder
import config

oai           = OpenAI(api_key=OPENAI_API_KEY)
qdrant        = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)
bm25_model    = SparseTextEmbedding(model_name="Qdrant/bm25")
cross_encoder = CrossEncoder(
    config.RERANK_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print("All clients ready.")

In [ ]:
# ── Generate GT pairs (only needed once — skipped if file exists) ─────────────
import os, json
from eval.generate_gt import generate_gt_pairs, load_gt_pairs

if os.path.exists(config.GT_PAIRS_PATH):
    print('GT pairs already exist — loading.')
    gt_pairs = load_gt_pairs()
else:
    # Generate from the recursive collection (chunking doesn't affect GT quality much)
    gt_pairs = generate_gt_pairs(
        oai=oai,
        qdrant=qdrant,
        collection_name='hr_rag_recursive',
    )

print(f'{len(gt_pairs)} GT pairs ready.')

In [ ]:
# ── Run the full evaluation grid ──────────────────────────────────────────────
from eval.evaluate import run_eval_grid, print_summary

df = run_eval_grid(
    gt_pairs=gt_pairs,
    oai=oai,
    qdrant=qdrant,
    bm25_model=bm25_model,
    cross_encoder=cross_encoder,
    openai_api_key=OPENAI_API_KEY,
    # Optionally limit scope during development:
    # chunking_strategies=['recursive', 'semantic'],
    # retrieval_strategies=['dense', 'hybrid'],
    include_rerank=True,
)

print_summary(df)

In [ ]:
# ── Pivot table: context_recall by chunking × retrieval ───────────────────────
import pandas as pd

pivot = df[~df['rerank']].pivot_table(
    index='chunking', columns='retrieval', values='context_recall'
).round(4)

print('Context recall (no rerank):')
print(pivot)

# Best overall config
best = df.loc[df['context_recall'].idxmax()]
print(f"\nBest config: {best['config']}")
print(f"  context_recall    : {best['context_recall']:.4f}")
print(f"  faithfulness      : {best['faithfulness']:.4f}")
print(f"  answer_relevancy  : {best['answer_relevancy']:.4f}")